# Modell 1 Baseline

### Was kann dieses Modell?

### Daten: Ein Rennen (Belgien)
### Modelle: LogReg, DTree, XGBoost
### Training: 80/20 Train-Test-Split
### Vorhersage: Alle Probs für alle Fahrer
### Problem: Massives Overfitting

## Imports

In [1]:
import fastf1
from fastf1 import plotting
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
import os

# sklearn related imports
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import brier_score_loss
from sklearn.utils.class_weight import compute_sample_weight



project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(project_root)
from utils.Feature_Engineer import FeatureEngineer
from utils.Metrics import Metrics
from utils.Models import Models
from utils.experiment_db import init_db
init_db()


pd.set_option('display.width', None)         # passt die Ausgabe-Breite an
pd.set_option('display.max_columns', None)   # zeigt alle Spalten
pd.set_option('display.max_rows', None)      # zeigt alle Reihen

%load_ext autoreload
%autoreload 2

## Rennen definieren

In [2]:
YEAR = 2024
GRAND_PRIX = ["Belgian Grand Prix"]
SESSION_TYPE = "R"

## Daten laden

In [3]:
all_laps = []
for gp in GRAND_PRIX:
    try:
        session = fastf1.get_session(YEAR, gp, SESSION_TYPE)
        session.load(weather=True, laps=True, telemetry=True)
        laps = session.laps.copy()
        weather = session.weather_data.copy()
        
        laps = laps.sort_values("Time")
        weather = weather.sort_values("Time")

        merged = pd.merge_asof(
                laps,
                weather,
                on="Time",
                direction="backward"
            )
        
        merged['GP'] = gp
        merged['Year'] = YEAR
        
        all_laps.append(merged)
        print(f"✔ Loaded {gp}")
        
    except Exception as e:
        print(f"Error loading {gp}: {e}")
        
data = pd.concat(all_laps, ignore_index=True)

req         WARNING 	DEFAULT CACHE ENABLED! (2.5 GB) /Users/lukaskoch/Library/Caches/fastf1
core           INFO 	Loading data for Belgian Grand Prix - Race [v3.7.0]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
core        WARNING 	Fixed incorrect tyre stint information for driver '14'
core        WARNING 	Fixed incorrect tyre stint information for driver '3'
core        WARNING 	Fixed incorrect tyre stint information for driver '18'
core        WARNING 	Fixed incorrect tyre stint information for driver '22'
req            INFO 	Using cached data for car_data
req            INFO 	

✔ Loaded Belgian Grand Prix


In [4]:
data.head()

,Time,Driver,DriverNumber,LapTime,LapNumber,Stint,PitOutTime,PitInTime,Sector1Time,Sector2Time,Sector3Time,Sector1SessionTime,Sector2SessionTime,Sector3SessionTime,SpeedI1,SpeedI2,SpeedFL,SpeedST,IsPersonalBest,Compound,TyreLife,FreshTyre,Team,LapStartTime,LapStartDate,TrackStatus,Position,Deleted,DeletedReason,FastF1Generated,IsAccurate,AirTemp,Humidity,Pressure,Rainfall,TrackTemp,WindDirection,WindSpeed,GP,Year
0,0 days 00:57:06.947000,LEC,16,0 days 00:01:51.912000,1.0,1.0,NaT,NaT,NaT,0 days 00:00:48.663000,0 days 00:00:29.571000,NaT,0 days 00:56:37.511000,0 days 00:57:07.067000,314.0,204.0,219.0,303.0,False,MEDIUM,1.0,True,Ferrari,0 days 00:55:14.764000,2024-07-28 13:03:52.742,1,1.0,False,,False,False,21.0,49.0,978.2,False,42.6,321,1.3,Belgian Grand Prix,2024
1,0 days 00:57:07.735000,HAM,44,0 days 00:01:52.700000,1.0,1.0,NaT,NaT,NaT,0 days 00:00:49.406000,0 days 00:00:29.478000,NaT,0 days 00:56:38.468000,0 days 00:57:07.904000,326.0,204.0,218.0,309.0,False,MEDIUM,1.0,True,Mercedes,0 days 00:55:14.764000,2024-07-28 13:03:52.742,1,2.0,False,,False,False,21.0,49.0,978.2,False,42.6,321,1.3,Belgian Grand Prix,2024
2,0 days 00:57:08.474000,PER,11,0 days 00:01:53.439000,1.0,1.0,NaT,NaT,NaT,0 days 00:00:50.038000,0 days 00:00:29.491000,NaT,0 days 00:56:39.019000,0 days 00:57:08.631000,316.0,201.0,216.0,310.0,False,MEDIUM,1.0,True,Red Bull Racing,0 days 00:55:14.764000,2024-07-28 13:03:52.742,1,3.0,False,,False,False,21.0,49.0,978.2,False,42.6,321,1.3,Belgian Grand Prix,2024
3,0 days 00:57:09.283000,PIA,81,0 days 00:01:54.248000,1.0,1.0,NaT,NaT,NaT,0 days 00:00:50.480000,0 days 00:00:29.643000,NaT,0 days 00:56:39.863000,0 days 00:57:09.531000,332.0,203.0,218.0,302.0,False,MEDIUM,1.0,True,McLaren,0 days 00:55:14.764000,2024-07-28 13:03:52.742,1,4.0,False,,False,False,21.0,49.0,978.2,False,42.6,321,1.3,Belgian Grand Prix,2024
4,0 days 00:57:09.798000,RUS,63,0 days 00:01:54.763000,1.0,1.0,NaT,NaT,NaT,0 days 00:00:51.125000,0 days 00:00:29.496000,NaT,0 days 00:56:40.426000,0 days 00:57:10.051000,331.0,202.0,218.0,310.0,False,MEDIUM,1.0,True,Mercedes,0 days 00:55:14.764000,2024-07-28 13:03:52.742,1,5.0,False,,False,False,21.0,49.0,978.2,False,42.6,321,1.3,Belgian Grand Prix,2024


## Einheitliche Features laden

In [5]:
fe = FeatureEngineer()
merged = fe.create_features(data)

## Modelle definieren

In [6]:
models = Models.define_models()

## Features auswählen

In [7]:
features = fe.choose_features(merged)
features

['gapAheadSeconds',
 's1_delta',
 's2_delta',
 's3_delta',
 'lap_pace_delta',
 'is_raining',
 'position_change',
 'compound_HARD',
 'compound_MEDIUM',
 'compound_SOFT',
 'status_1',
 'status_12']

In [8]:
X, y = fe.split_features(merged, features)

## Plot Funktionen

In [9]:
import math
import matplotlib.pyplot as plt
import fastf1.plotting as f1plot

In [10]:
def plot_pit_stop_probabilities(merged, model_name):

    f1plot.setup_mpl(mpl_timedelta_support=True, color_scheme='fastf1')

    # Liste aller Fahrer im Rennen
    drivers = merged['Driver'].unique()
    drivers = [d for d in drivers if isinstance(d, str)]  # falls NaN vorkommen

    # Grid-Einstellungen
    rows, cols = 4, 5
    plots_per_page = rows * cols
    num_pages = math.ceil(len(drivers) / plots_per_page)

    for page in range(num_pages):

        fig, axes = plt.subplots(rows, cols, figsize=(22, 14))
        axes = axes.flatten()

        # Fahrer für diese Seite wählen
        start = page * plots_per_page
        end = start + plots_per_page
        driver_subset = drivers[start:end]

        for ax, driver in zip(axes, driver_subset):

            drv = merged[merged['Driver'] == driver]

            # Stil aus FastF1 holen
            try:
                style = plotting.get_driver_style(
                    identifier=driver, 
                    style=['color', 'linestyle'], 
                    session=session
                )
            except:
                style = {"color": "white", "linestyle": "-"}

            # Probability Plot
            ax.plot(
                drv['LapNumber'],
                drv['pit_stop_prob'],
                label=f'{driver}',
                **style
            )
            #ax.set_yscale('log')

            # Pit Stops markieren
            #pit_laps = drv.loc[drv['Pit_next_lap'].astype(int) == 1, 'LapNumber']
            pit_laps = drv.loc[drv['pit_this_lap'] == 1, 'LapNumber']

            for lap in pit_laps:
                ax.axvline(lap, color='red', linestyle='--', alpha=0.7)

            ax.set_title(f'{driver}')
            ax.set_xlabel('Lap')
            ax.set_ylabel('Pit Stop Prob (log)')

        # Leere Achsen verstecken (falls letzte Seite < 12 Fahrer)
        for ax in axes[len(driver_subset):]:
            ax.axis('off')
            
        fig.suptitle(f"Pit Stop Probability — {model_name}", fontsize=24)
        plt.tight_layout()
        
        save_path = f"plots/plots_model_01/all_{model_name}.png"
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        fig.savefig(save_path, dpi=300, bbox_inches="tight")
        plt.close(fig)


In [11]:
def plot_albon_pit_stop_probability(merged, session, model_name):
    fastf1.plotting.setup_mpl(mpl_timedelta_support=True, color_scheme='fastf1')

    albon = merged[merged['Driver'] == 'ALB']
    style = plotting.get_driver_style(identifier='ALB', style=['color', 'linestyle'], session=session)
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.plot(albon['LapNumber'], albon['pit_stop_prob'], label='Predicted Pit Stop Probability', **style)
    #ax.set_yscale('log')

    pit_laps = albon.loc[albon['pit_this_lap'] == 1, 'LapNumber']


    for lap in pit_laps:
        ax.axvline(lap, color='red', linestyle='--', alpha=0.7)
        ax.text(lap, ax.get_ylim()[1]*0.8, f'',
                rotation=90, color='red', fontsize=9, ha='center', va='bottom') 

    ax.set_title('Predicted Pit Stop Probability for ALB in 2024 Spa GP')
    ax.set_xlabel('Lap Number')
    ax.set_ylabel('Pit Stop Probability (log scale)')
    max_lap = int(albon['LapNumber'].max())
    ax.set_xticks(range(0, max_lap + 1, 2))
    ax.legend()
    plt.tight_layout()
    
    
    save_path = f"plots/plots_model_01/albon_{model_name}.png"
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.close()
    #plt.show()

## Modelle laufen lassen

In [12]:
from utils.experiment_db import log_or_update_metrics

def train_and_evaluate(model, X, y, model_name):
    print(f"\n🔧 Training {model_name} ...")

    # 1️⃣ Split
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
    
    # 2️⃣ Trainieren
    model.fit(X_train, y_train)

    # 3️⃣ Vorhersagen
    y_pred_prob = model.predict_proba(X_test)[:, 1]
    
    # 4️⃣ plotten
    merged["pit_stop_prob"] = model.predict_proba(X)[:, 1]
    #plot_albon_pit_stop_probability(merged, session, model_name)
    #plot_pit_stop_probabilities(merged, model_name)

    # 5️⃣ Alle Metriken berechnen
    
    metrics = Metrics.calculate_metrics(f"approach_01_{model_name}", y_test, y_pred_prob)
    Metrics.print_metrics(metrics)
    
    
    log_or_update_metrics(
        model_name=f"approach_01_{model_name}",
        metrics={
            **metrics
        }
    )
    
    return metrics

In [13]:
scores = {}
for name, model in models.items():
    metrics = train_and_evaluate(model, X, y, name)
    scores[name] = metrics


🔧 Training LogisticRegression ...
   ✔ Brier-Score:              0.2172 -> optimal 0.0
   ✔ Log-Loss:                 0.6188 -> optimal 0.0
   ✔ ROC-AUC:                  0.6781 -> optimal 1.0
   ✔ PR-AUC:                   0.1814 -> optimal 1.0

🔧 Training DecisionTree ...
   ✔ Brier-Score:              0.1906 -> optimal 0.0
   ✔ Log-Loss:                 1.0875 -> optimal 0.0
   ✔ ROC-AUC:                  0.6151 -> optimal 1.0
   ✔ PR-AUC:                   0.0759 -> optimal 1.0

🔧 Training xgboost ...
   ✔ Brier-Score:              0.0393 -> optimal 0.0
   ✔ Log-Loss:                 0.1588 -> optimal 0.0
   ✔ ROC-AUC:                  0.7619 -> optimal 1.0
   ✔ PR-AUC:                   0.1028 -> optimal 1.0
